In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/forage.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1.4':'20.81']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.4':'20.81'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.4':'20.81'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 16:45:21,665 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 16:45:21,671 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import mlp_optimized

mlp_model = mlp_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, 
                          aim='classification', 
                          hidden_layer_sizes=(64,32),
                          activation='tanh',
                          learning_rate='adaptive',
                          max_iter=10,
                          random_state=1)
mlp_model[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,MLP,0.985185,0.978947,1.0,"[[40, 0], [2, 93]]",1.0,1.0,1.0,"[[18, 0], [0, 42]]"


## Definição das Zonas Espectrais

In [3]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Al', 1.40, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.16),
('S', 2.16, 2.44),
('Rh L + Ar', 2.44, 3.10),
('K', 3.10, 3.46),
('Ca ka', 3.46, 3.86),
('Ca kb', 3.86, 4.37),
('Ti ka', 4.37, 4.66),
('Ti kb', 4.66, 5.08),
('background1', 5.08, 5.72),
('Mn', 5.72, 6.10),
('Fe ka', 6.10, 6.76),
('Fe kb', 6.76, 7.20),
('Ni', 7.20, 7.69),
('Cu', 7.69, 8.45),
('Zn', 8.45, 8.81),
('background2', 8.81, 13.10),
('sum Fe' , 13.10, 13.63),
('background3', 13.63, 18.0),
('Compton scattering', 18.0, 19.70),
('Rayleight scattering', 19.70, 20.80)
]

## SHAP

In [4]:
shap_global_importance = pd.read_csv('shap_forage.csv', sep=';') # loading previously saved shap_unique_df
# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,3.70,0.034479,Ca ka
1,6.40,0.030021,Fe ka
2,3.36,0.020017,K
3,1.74,0.008024,Si
4,4.00,0.006471,Ca kb
5,4.50,0.004159,Ti ka
6,2.56,0.002085,Rh L + Ar
7,17.43,0.000544,background3
8,5.92,0.000510,Mn
9,10.55,0.000498,background2


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [5]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zone 'Al': VE = 92.81%, dim = 12 variables
Zone 'Si': VE = 99.42%, dim = 12 variables
Zone 'P': VE = 94.16%, dim = 16 variables
Zone 'S': VE = 89.31%, dim = 15 variables
Zone 'Rh L + Ar': VE = 95.85%, dim = 34 variables
Zone 'K': VE = 98.44%, dim = 19 variables
Zone 'Ca ka': VE = 89.46%, dim = 21 variables
Zone 'Ca kb': VE = 96.25%, dim = 26 variables
Zone 'Ti ka': VE = 98.15%, dim = 15 variables
Zone 'Ti kb': VE = 76.25%, dim = 22 variables
Zone 'background1': VE = 66.63%, dim = 33 variables
Zone 'Mn': VE = 94.16%, dim = 20 variables
Zone 'Fe ka': VE = 99.38%, dim = 34 variables
Zone 'Fe kb': VE = 94.04%, dim = 23 variables
Zone 'Ni': VE = 77.76%, dim = 25 variables
Zone 'Cu': VE = 84.25%, dim = 39 variables
Zone 'Zn': VE = 82.20%, dim = 19 variables
Zone 'background2': VE = 86.56%, dim = 215 variables
Zone 'sum Fe': VE = 88.47%, dim = 27 variables
Zone 'background3': VE = 87.92%, dim = 219 variables
Zone 'Compton scattering': VE = 88.73%, dim = 85 variables
Zone 'Rayleight scattering

In [6]:
y_pred = mlp_model[4]['MLP'].values # using the continuous predictions from SVM, extracting as 1D array
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred
)

In [7]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_pred = pd.Series(mlp_model[4]['MLP'].values) # using the continuous predictions from SVM, extracting as 1D array

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 133 | Discarded: 43
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Computing Covariance for Predicates
Metric: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Ba

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Fe ka > -3.05,Fe ka > -3.05,Fe ka > -3.05,Fe ka > -3.05
1,Fe ka > -3.74,Fe ka > -3.74,Fe ka > -3.74,Fe ka > -3.74
2,Fe ka > -2.26,Fe ka > -2.26,Fe ka > -2.26,Fe ka > -2.26
3,K > -6.69,K > -6.69,K > -6.69,K > -6.69
4,Ca ka <= 3.95,Ca ka <= 3.95,Ca ka <= 3.95,Ca ka <= 3.95
...,...,...,...,...
124,Zn > -0.37,Class_A,sum Fe <= 0.62,Zn > 0.10
125,Class_A,Class_B,background2 <= 1.51,Zn <= -0.11
126,Class_B,NaN,Class_A,Fe ka <= -3.05
127,NaN,NaN,Class_B,Class_A


In [8]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -3.05,21.730280,Fe ka,-3.05,>
1,K > -6.69,11.048593,K,-6.69,>
2,Ca ka <= 3.95,8.618784,Ca ka,3.95,<=
3,Si > -2.51,7.521654,Si,-2.51,>
4,Ti ka > -0.70,2.030123,Ti ka,-0.70,>
5,Fe kb > -0.63,1.836228,Fe kb,-0.63,>
6,Ca kb <= 1.57,1.566439,Ca kb,1.57,<=
7,Rh L + Ar > -0.77,1.434719,Rh L + Ar,-0.77,>
8,background3 > 0.14,1.241904,background3,0.14,>
9,Compton scattering > -0.40,0.975374,Compton scattering,-0.40,>


In [9]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zone 'Al': VE = 95.14%, dim = 12 variables
Zone 'Si': VE = 99.71%, dim = 12 variables
Zone 'P': VE = 97.48%, dim = 16 variables
Zone 'S': VE = 93.90%, dim = 15 variables
Zone 'Rh L + Ar': VE = 96.86%, dim = 34 variables
Zone 'K': VE = 99.17%, dim = 19 variables
Zone 'Ca ka': VE = 93.72%, dim = 21 variables
Zone 'Ca kb': VE = 98.07%, dim = 26 variables
Zone 'Ti ka': VE = 98.49%, dim = 15 variables
Zone 'Ti kb': VE = 76.99%, dim = 22 variables
Zone 'background1': VE = 66.94%, dim = 33 variables
Zone 'Mn': VE = 95.57%, dim = 20 variables
Zone 'Fe ka': VE = 99.73%, dim = 34 variables
Zone 'Fe kb': VE = 94.82%, dim = 23 variables
Zone 'Ni': VE = 79.28%, dim = 25 variables
Zone 'Cu': VE = 84.38%, dim = 39 variables
Zone 'Zn': VE = 82.21%, dim = 19 variables
Zone 'background2': VE = 86.54%, dim = 215 variables
Zone 'sum Fe': VE = 88.64%, dim = 27 variables
Zone 'background3': VE = 87.63%, dim = 219 variables
Zone 'Compton scattering': VE = 89.25%, dim = 85 variables
Zone 'Rayleight scattering

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > -3.05,21.730280,Fe ka,-3.05,>,-11.993967,102.0,0.012499,Fe ka > -11.993967
1,Fe ka > -3.74,18.176011,Fe ka,-3.74,>,-14.394181,68.0,0.017162,Fe ka > -14.394181
2,Fe ka > -2.26,14.346492,Fe ka,-2.26,>,-8.585461,77.0,0.009247,Fe ka > -8.585461
3,K > -6.69,11.048593,K,-6.69,>,-65.671272,79.0,0.023761,K > -65.671272
4,Ca ka <= 3.95,8.618784,Ca ka,3.95,<=,24.763583,89.0,0.045885,Ca ka <= 24.763583
...,...,...,...,...,...,...,...,...,...
125,Zn > 0.10,0.221977,Zn,0.10,>,0.231960,7.0,0.000820,Zn > 0.231960
126,background1 <= -0.08,0.215201,background1,-0.08,<=,-0.162772,3.0,0.007259,background1 <= -0.162772
127,Rayleight scattering <= 0.53,0.198331,Rayleight scattering,0.53,<=,0.681638,73.0,0.012780,Rayleight scattering <= 0.681638
128,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [10]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Fe ka':
  - Dimensão: 34 variáveis espectrais
  - Range de energias: 6.1 - 6.76
  - Variância explicada pela PC1: 99.73%


In [11]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=mlp_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 133 | Discarded: 43
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 132 | Discarded: 44
PERTURBATION IMPORTANCE FOR PREDICATES
Task type (aim): classification
Perturbation mode: median
Statistics source: full
Metric: probability_shift
Total folds: 10


[Bag_1] Processing 132 predicates...
  Predicate: Al > -0.68 (n=87)
    Zone: 1

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,K > 2.15,K > 2.15,Ca ka <= -1.50,Ca ka <= -1.50
1,K <= -1.46,Ca ka <= -1.50,K > 2.15,K > -1.46
2,Ca ka <= 1.77,K <= -1.46,Ca ka <= 1.77,Ca ka > 1.77
3,K > -1.46,Ca ka > -4.65,K > -1.46,K > 2.15
4,Ca ka <= -1.50,K > -1.46,Ca ka > 1.77,K > -6.69
...,...,...,...,...
131,Zn > -0.11,Zn > 0.10,Zn > 0.10,Zn > -0.37
132,Zn > 0.10,Class_A,Zn > -0.11,Zn > -0.11
133,Class_A,Class_B,Class_A,Zn > 0.10
134,Class_B,NaN,Class_B,Class_A


In [12]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ca ka <= -1.50,0.041004
1,K > 2.15,0.037478
2,Ca ka > 1.77,0.033290
3,K <= -1.46,0.032802
4,Ca ka <= 1.77,0.031893
...,...,...
127,Zn <= 0.10,0.000835
128,Zn <= 0.34,0.000691
129,Zn > -0.37,0.000517
130,Zn > -0.11,0.000472


In [13]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,K > 2.15,6.785412,K,2.15,>
1,Ca ka <= -1.50,6.421293,Ca ka,-1.50,<=
2,Fe ka <= -3.05,2.173922,Fe ka,-3.05,<=
3,Si <= -1.67,1.540259,Si,-1.67,<=
4,Mn <= -0.41,1.296415,Mn,-0.41,<=
5,Ca kb <= -0.59,1.240538,Ca kb,-0.59,<=
6,S > 0.55,1.198419,S,0.55,>
7,background3 > 0.14,1.116352,background3,0.14,>
8,Rh L + Ar > -0.16,0.996417,Rh L + Ar,-0.16,>
9,Fe kb <= -0.91,0.958183,Fe kb,-0.91,<=


In [14]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,K > 2.15,6.785412,K,2.15,>,20.676485,90.0,0.022977,K > 20.676485
1,Ca ka <= -1.50,6.421293,Ca ka,-1.50,<=,-9.476439,94.0,0.083086,Ca ka <= -9.476439
2,K > -1.46,5.136089,K,-1.46,>,-12.506148,111.0,0.014010,K > -12.506148
3,Ca ka <= 1.77,4.949388,Ca ka,1.77,<=,11.953266,82.0,0.013408,Ca ka <= 11.953266
4,K <= -1.46,4.860323,K,-1.46,<=,-12.506148,111.0,0.014010,K <= -12.506148
...,...,...,...,...,...,...,...,...,...
132,Zn > -0.37,0.004708,Zn,-0.37,>,-0.825949,38.0,0.000121,Zn > -0.825949
133,Zn > -0.11,0.002687,Zn,-0.11,>,-0.257461,132.0,0.004205,Zn > -0.257461
134,Zn > 0.10,0.001628,Zn,0.10,>,0.231960,7.0,0.000820,Zn > 0.231960
135,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


In [15]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'K':
  - Dimensão: 19 variáveis espectrais
  - Variância explicada pela PC1: 99.17%


In [16]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = mlp_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = mlp_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,3.36,0.011790,K
1,6.4,0.010076,Fe ka
2,3.66,0.009841,Ca ka
3,1.74,0.004149,Si
4,4,0.004139,Ca kb
5,4.5,0.004036,Ti ka
6,2,0.003172,P
7,5.86,0.003121,Mn
8,2.62,0.003010,Rh L + Ar
9,7.08,0.001923,Fe kb


In [17]:
import numpy as np

max_len = max(
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ca ka,K,K,Fe ka
1,Fe ka,Fe ka,Ca ka,K
2,K,Ca ka,Fe ka,Ca ka
3,Si,Si,Si,Si
4,Ca kb,Ca kb,Mn,Ti ka
5,Ti ka,Ti ka,Ca kb,Fe kb
6,Rh L + Ar,P,S,Ca kb
7,background3,Mn,background3,Rh L + Ar
8,Mn,Rh L + Ar,Rh L + Ar,background3
9,background2,Fe kb,Fe kb,Compton scattering


In [18]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_13228\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
3,Permutation,LRC_perturbation,0.821418
4,Permutation,LRC_covariance,0.631460
0,Shap,Permutation,0.551191
2,Shap,LRC_covariance,0.531470
1,Shap,LRC_perturbation,0.523339
5,LRC_perturbation,LRC_covariance,0.509926


In [19]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')